# Sinkhorn potentials along the iteration

This notebook generates `fig:sinkhorn-potentials-iterations`.  It uses the same one-dimensional histograms and KL-normalized potential convention as `fig:sinkhorn-dual-potentials-epsilon`, but fixes `epsilon` and follows the alternating row/column scalings through selected iterations.

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/mpl-ot4ml")

for candidate in [Path.cwd(), Path.cwd() / "notebooks-figures", Path.cwd().parent / "notebooks-figures"]:
    if (candidate / "figure_style.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise RuntimeError("Could not locate figure_style.py")

import matplotlib.pyplot as plt
import numpy as np
import ot
from matplotlib.collections import LineCollection
from matplotlib.colors import LinearSegmentedColormap, to_rgb
from matplotlib.patches import Polygon

from figure_style import (
    BLUE,
    DIRAC_MARKER_SIZE,
    GRAY,
    LIGHT_GRAY,
    ORANGE,
    RED,
    VIOLET,
    box_axes,
    canonical_matching_clouds,
    figure_dir,
    interp_color,
    padded_limits,
    remove_axes,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()
np.random.seed(0)

def normal_pdf(x, mean, std):
    return np.exp(-0.5 * ((x - mean) / std) ** 2) / (std * np.sqrt(2 * np.pi))


def mixture_pdf(x, weights, means, stds):
    density = np.zeros_like(x, dtype=float)
    for weight, mean, std in zip(weights, means, stds):
        density += weight * normal_pdf(x, mean, std)
    return density


def sinkhorn_histograms(n=170):
    grid = np.linspace(-3.25, 3.15, n)
    alpha_density = mixture_pdf(grid, [0.58, 0.42], [-1.95, -0.10], [0.34, 0.54])
    beta_density = mixture_pdf(grid, [0.42, 0.58], [-0.75, 1.55], [0.42, 0.36])
    a = alpha_density / alpha_density.sum()
    b = beta_density / beta_density.sum()
    C = (grid[:, None] - grid[None, :]) ** 2
    C = C / np.median(C[C > 0])
    return grid, alpha_density, beta_density, a, b, C


def kl_normalized_solution(a, b, C, epsilon, *, num_iter=10000):
    K = (a[:, None] * b[None, :]) * np.exp(-C / epsilon)
    plan, log = ot.sinkhorn(
        a,
        b,
        K,
        reg=1.0,
        method="sinkhorn_log",
        log=True,
        numItermax=num_iter,
        stopThr=1e-12,
    )
    f = epsilon * np.asarray(log["log_u"])
    g = epsilon * np.asarray(log["log_v"])
    shift = -float(np.dot(a, f))
    f = f + shift
    g = g - shift
    return plan, f, g

## Iterative logarithmic scalings

At iteration `k`, the current coupling has the form `diag(u) K diag(v)`, where `K=(a otimes b) exp(-C/epsilon)`.  The plotted potentials are `epsilon log u` and `epsilon log v`, with the same gauge in every panel.

In [ ]:
fig_name = "sinkhorn-potentials-iterations"
out = figure_dir(fig_name)

grid, alpha_density, beta_density, a, b, C = sinkhorn_histograms(n=170)
epsilon = 0.045
K = (a[:, None] * b[None, :]) * np.exp(-C / epsilon)
selected = [0, 1, 3, 12]

u = np.ones_like(a)
v = np.ones_like(b)
states = {}
for k in range(max(selected) + 1):
    f = epsilon * np.log(np.maximum(u, 1e-300))
    g = epsilon * np.log(np.maximum(v, 1e-300))
    shift = -float(np.dot(a, f))
    f = f + shift
    g = g - shift
    if k in selected:
        states[k] = (f.copy(), g.copy())
    u = a / np.maximum(K @ v, 1e-300)
    v = b / np.maximum(K.T @ u, 1e-300)

all_values = np.concatenate([np.r_[f, g] for f, g in states.values()])
ymin, ymax = np.percentile(all_values, [1, 99])
pad = 0.16 * (ymax - ymin)
ymin, ymax = ymin - pad, ymax + pad
hist_scale = 0.24 * (ymax - ymin)
hist_base = ymin

## Exported iteration panels

All panels share the same axes.  The faint silhouettes provide spatial context for the red source and blue target histograms.

In [ ]:
def draw_state(k, filename):
    f, g = states[k]
    fig, ax = plt.subplots(figsize=(2.36, 1.84))
    ax.fill_between(grid, hist_base, hist_base + hist_scale * alpha_density / alpha_density.max(), color=RED, alpha=0.14, linewidth=0, zorder=1)
    ax.fill_between(grid, hist_base, hist_base - hist_scale * beta_density / beta_density.max(), color=BLUE, alpha=0.14, linewidth=0, zorder=1)
    ax.plot(grid, f, color=RED, lw=1.14, zorder=3)
    ax.plot(grid, g, color=BLUE, lw=1.14, zorder=3)
    ax.set_xlim(grid.min(), grid.max())
    ax.set_ylim(ymin, ymax)
    ax.set_xticks([])
    ax.set_yticks([])
    box_axes(ax)
    save_pdf(fig, out / filename, pad_inches=0.055)
    plt.close(fig)


for k in selected:
    draw_state(k, f"iter-{k}.pdf")